In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\atalb\Documents\Coding\DeepLearning\Datasets\FakeNews\train.csv")

In [2]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [3]:
df = df.dropna()

In [4]:
X=df.drop("label",axis=1)
y=df["label"]

# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [5]:
X.shape

(18285, 4)

In [6]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [7]:
## Vocabulary size
voc_size=5000

In [8]:
## One hot representation
messages=X.copy()
messages.reset_index(inplace=True)

In [11]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()
    
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [12]:
one_hot_rep=[one_hot(words,voc_size)for words in corpus]
one_hot_rep

[[2759, 1488, 2792, 2437, 4369, 488, 1792, 3713, 4173, 3779],
 [1804, 2181, 97, 3338, 1855, 2257, 730],
 [374, 2861, 3059, 3373],
 [2237, 4186, 3303, 2272, 4834, 3838],
 [2584, 1855, 4366, 4603, 2722, 1446, 1855, 3799, 2478, 200],
 [4766,
  2217,
  2053,
  1910,
  3561,
  318,
  4935,
  1702,
  2408,
  4255,
  263,
  3023,
  3966,
  797,
  730],
 [4790, 248, 4693, 226, 628, 1913, 4599, 1775, 2526, 445, 4438],
 [4807, 2470, 1779, 941, 591, 2200, 318, 2712, 2526, 445, 4438],
 [736, 4635, 1277, 3050, 1833, 4181, 1301, 235, 318, 1152],
 [988, 4085, 85, 1478, 888, 2824, 3926, 4359],
 [4842, 613, 2730, 2092, 1893, 2519, 4169, 2728, 4489, 3453, 4274],
 [2272, 3042, 4369, 4181, 318, 591],
 [2919, 880, 447, 2107, 799, 3088, 4960, 552, 3405],
 [4581, 2027, 1112, 908, 4674, 2482, 4480, 2526, 445, 4438],
 [4352, 1039, 3005, 1450, 1444, 2526, 445, 4438],
 [2570, 1379, 3466, 38, 4510, 699, 1683, 3698, 2835, 291],
 [2881, 4651, 2181],
 [2617, 2055, 1944, 2184, 318, 4960, 377, 730],
 [815, 2041, 97, 3

In [13]:
sent_length=20
embedded_docs=pad_sequences(one_hot_rep,padding='pre',maxlen=sent_length)
print(embedded_docs)

[[   0    0    0 ... 3713 4173 3779]
 [   0    0    0 ... 1855 2257  730]
 [   0    0    0 ... 2861 3059 3373]
 ...
 [   0    0    0 ... 2526  445 4438]
 [   0    0    0 ...  532 2053 1128]
 [   0    0    0 ... 4409 1929 4585]]


In [14]:
## creation of model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 20, 40)            200000    
                                                                 
 lstm (LSTM)                 (None, 100)               56400     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 256501 (1001.96 KB)
Trainable params: 256501 (1001.96 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [15]:
len(embedded_docs), y.shape

(18285, (18285,))

In [16]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

In [18]:
## model training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10


192/192 [==============================] - 13s 35ms/step - loss: 0.3287 - accuracy: 0.8473 - val_loss: 0.2032 - val_accuracy: 0.9152
Epoch 2/10
192/192 [==============================] - 5s 27ms/step - loss: 0.1387 - accuracy: 0.9468 - val_loss: 0.2131 - val_accuracy: 0.9175
Epoch 3/10
192/192 [==============================] - 5s 28ms/step - loss: 0.0908 - accuracy: 0.9672 - val_loss: 0.2197 - val_accuracy: 0.9067
Epoch 4/10
192/192 [==============================] - 6s 34ms/step - loss: 0.0593 - accuracy: 0.9793 - val_loss: 0.2750 - val_accuracy: 0.9180
Epoch 5/10
192/192 [==============================] - 6s 30ms/step - loss: 0.0321 - accuracy: 0.9891 - val_loss: 0.3196 - val_accuracy: 0.9075
Epoch 6/10
192/192 [==============================] - 6s 29ms/step - loss: 0.0208 - accuracy: 0.9931 - val_loss: 0.3671 - val_accuracy: 0.9142
Epoch 7/10
192/192 [==============================] - 6s 31ms/step - loss: 0.0135 - accuracy: 0.9954 - val_loss: 0.3843 - val_accuracy: 0.9

In [ ]:
# ## adding dropout
# from tesorflow.keras.layers import Dropout
# # model.add(Dropout(0.3))
# #creating model
# embedding_vector_features=40
# model=Sequential()
# model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
# model.add(Dropout(0.3))
# model.add(LSTM(100))
# model.add(Dropout(0.3))
# model.add(Dense(1,activation='sigmoid'))
# model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
# print(model.summary())

In [21]:
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np

y_pred_probs = model.predict(X_test)          # raw probabilities, e.g. [0.83, 0.12, ...]
y_pred = (y_pred_probs > 0.5).astype(int)     # convert → [1, 0, ...]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

189/189 [==============================] - 2s 9ms/step
Accuracy: 0.9093620546810274
Confusion Matrix:
 [[3133  286]
 [ 261 2355]]


In [22]:
accuracy_score(y_test, y_pred)

0.9093620546810274